## 일별 박스오피스 현황에 대한 정보 수집
- 영화 진흥위원회에서 관리하는 api를 이용

### 1. 라이브러리 호출

In [1]:
import math
import time
from getpass import getpass                     # 인증키가 화면에 표시되지 않도록 하는 라이브러리
from urllib.parse import unquote                # URL을 보기 좋게 만들어주는 라이브러리

import requests
import pandas as pd

### 2. 인증키 입력

In [2]:
general_key = getpass("공공데이터포털 인증키를 입력하세요: ").strip()

SERVICE_KEY = unquote(general_key)

## 3. API 주소와 기본 요청 변수 설정

In [3]:
API_URL = ("http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json")

# 한 번 호출할 때 가져오는 데이터 수
NUM_OF_ROWS = 10
TARGET_DT = 20260713


## 4. 날짜 하나 가져와서 연습해보기

In [4]:
params = {
    "key" : SERVICE_KEY,
    "itemPerPage" : NUM_OF_ROWS,
    "targetDt" : TARGET_DT
}

response = requests.get(API_URL, params = params, timeout = 30)

In [5]:
print("HTTP 상태코드: ", response.status_code)
print("응답 형식: ", response.headers.get("Content-Type"))

HTTP 상태코드:  200
응답 형식:  application/json;charset=utf-8


In [6]:
api_data = response.json()

In [7]:
api_data

{'boxOfficeResult': {'boxofficeType': '일별 박스오피스',
  'showRange': '20260713~20260713',
  'dailyBoxOfficeList': [{'rnum': '1',
    'rank': '1',
    'rankInten': '0',
    'rankOldAndNew': 'OLD',
    'movieCd': '20259946',
    'movieNm': '모아나',
    'openDt': '2026-07-08',
    'salesAmt': '322845250',
    'salesShare': '28.1',
    'salesInten': '-1501529020',
    'salesChange': '-82.3',
    'salesAcc': '5962089030',
    'audiCnt': '29084',
    'audiInten': '-135843',
    'audiChange': '-82.4',
    'audiAcc': '540540',
    'scrnCnt': '1127',
    'showCnt': '4097'},
   {'rnum': '2',
    'rank': '2',
    'rankInten': '1',
    'rankOldAndNew': 'OLD',
    'movieCd': '20242402',
    'movieNm': '눈동자',
    'openDt': '2026-06-24',
    'salesAmt': '287804770',
    'salesShare': '25.0',
    'salesInten': '-751204380',
    'salesChange': '-72.3',
    'salesAcc': '13397040300',
    'audiCnt': '26897',
    'audiInten': '-68680',
    'audiChange': '-71.9',
    'audiAcc': '1304135',
    'scrnCnt': '898',
 

In [8]:
# 1. 중간 딕셔너리 먼저 꺼내기
box_office = api_data.get("boxOfficeResult", {})

# 2. 각 항목 꺼내기
target_date = box_office.get("showRange", "")
daily_list = box_office.get("dailyBoxOfficeList", [])

print("조회 날짜:", target_date)
print("영화 목록 수:", len(daily_list))

조회 날짜: 20260713~20260713
영화 목록 수: 10


- 공공데이터 api 쓸때랑 형태가 다르네
- 확실히 json형태에서 파이썬의 딕셔너리와 유사한 형태로 변환한 후 데이터가 어떻게 축적되어있나 확인 후 필요한 부분들을 가져와주면 되네
- daily_list에 현재 영화 하나에 대한 정보가 쫘라라라ㅏㄱ 있을 텐데 각각의 순위만 따로 가져가고, 영화 이름만 가져가고 싶으면 for문을 이용해서 각각 가져간 후 데이터 프레임에 넣는 방법이 당장 생각나는 방법
- 디벨롭 할 수 있는 방안으로는 강사님이 해주셨던 거 차용해서 생각해보기

In [9]:
daily_list

[{'rnum': '1',
  'rank': '1',
  'rankInten': '0',
  'rankOldAndNew': 'OLD',
  'movieCd': '20259946',
  'movieNm': '모아나',
  'openDt': '2026-07-08',
  'salesAmt': '322845250',
  'salesShare': '28.1',
  'salesInten': '-1501529020',
  'salesChange': '-82.3',
  'salesAcc': '5962089030',
  'audiCnt': '29084',
  'audiInten': '-135843',
  'audiChange': '-82.4',
  'audiAcc': '540540',
  'scrnCnt': '1127',
  'showCnt': '4097'},
 {'rnum': '2',
  'rank': '2',
  'rankInten': '1',
  'rankOldAndNew': 'OLD',
  'movieCd': '20242402',
  'movieNm': '눈동자',
  'openDt': '2026-06-24',
  'salesAmt': '287804770',
  'salesShare': '25.0',
  'salesInten': '-751204380',
  'salesChange': '-72.3',
  'salesAcc': '13397040300',
  'audiCnt': '26897',
  'audiInten': '-68680',
  'audiChange': '-71.9',
  'audiAcc': '1304135',
  'scrnCnt': '898',
  'showCnt': '3214'},
 {'rnum': '3',
  'rank': '3',
  'rankInten': '-1',
  'rankOldAndNew': 'OLD',
  'movieCd': '20259781',
  'movieNm': '토이 스토리 5',
  'openDt': '2026-06-17',
  's